<a href="https://colab.research.google.com/github/pmudproject/analysis-equity-volatility/blob/main/NBAplayoffs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NBA PLAYOFF PREDICTION MODEL
# Logistic Regression + Playoff Dashboard
# ============================================================

# -----------------------------
# STEP 1 — INSTALL PACKAGES
# -----------------------------

!pip install nba_api pandas scikit-learn seaborn matplotlib


# -----------------------------
# STEP 2 — IMPORT LIBRARIES
# -----------------------------

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from nba_api.stats.endpoints import leaguegamefinder

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# -----------------------------
# STEP 3 — DOWNLOAD NBA DATA
# -----------------------------

gamefinder = leaguegamefinder.LeagueGameFinder(
    league_id_nullable='00'
)

games = gamefinder.get_data_frames()[0]

# Keep last 10 seasons
games = games[games['SEASON_ID'] >= '22014']


# -----------------------------
# STEP 4 — CLEAN + FEATURE ENGINEERING
# -----------------------------

# Convert win/loss to binary target
games['TARGET'] = games['WL'].map({
    'W':1,
    'L':0
})

# Home game indicator
games['HOME'] = games['MATCHUP'].apply(
    lambda x: 1 if 'vs.' in x else 0
)

# Convert dates
games['GAME_DATE'] = pd.to_datetime(
    games['GAME_DATE']
)

# Sort values
games = games.sort_values(
    ['TEAM_ID','GAME_DATE']
)

# Rest days
games['REST_DAYS'] = games.groupby(
    'TEAM_ID'
)['GAME_DATE'].diff().dt.days

# Rolling offensive production
games['ROLLING_PTS'] = games.groupby(
    'TEAM_ID'
)['PTS'].transform(
    lambda x: x.shift(1).rolling(
        10,
        min_periods=1
    ).mean()
)

# Rolling win percentage
games['ROLLING_WIN_PCT'] = games.groupby(
    'TEAM_ID'
)['TARGET'].transform(
    lambda x: x.shift(1).rolling(
        10,
        min_periods=1
    ).mean()
)

# Back-to-back indicator
games['B2B'] = (
    games['REST_DAYS'] <= 1
).astype(int)

# Features
features = [
    'HOME',
    'REST_DAYS',
    'ROLLING_PTS',
    'ROLLING_WIN_PCT',
    'B2B'
]

# Remove missing rows
model_data = games.dropna(
    subset=features + ['TARGET']
)

# X and y
X = model_data[features]

y = model_data['TARGET']


# -----------------------------
# STEP 5 — TRAIN LOGISTIC MODEL
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train,y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.5990824069172401


In [ ]:
# ============================================================
# BEST-OF-7 PLAYOFF SERIES SIMULATOR
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------
# FUNCTION TO PREDICT MATCHUP PROBABILITY
# ------------------------------------------------

def predict_matchup(
    home_team_abbr,
    away_team_abbr
):
    # Get the latest rolling stats for the home team
    home_team_stats = games[games['TEAM_ABBREVIATION'] == home_team_abbr].tail(1)
    # Get the latest rolling stats for the away team
    away_team_stats = games[games['TEAM_ABBREVIATION'] == away_team_abbr].tail(1)

    # Handle cases where team stats might not be found or are empty after tail(1)
    if home_team_stats.empty or away_team_stats.empty:
        # For simplicity, return 0.5 (even probability) if stats are missing
        return 0.5

    # Extract the rolling stats, checking for NaN
    rolling_pts_home = home_team_stats['ROLLING_PTS'].values[0]
    rolling_win_pct_home = home_team_stats['ROLLING_WIN_PCT'].values[0]

    # If any of the crucial rolling stats are NaN, return 0.5 as a fallback
    if pd.isna(rolling_pts_home) or pd.isna(rolling_win_pct_home):
        return 0.5

    # Create a hypothetical matchup entry for prediction
    # 'HOME' is 1 as we are predicting for the home team
    # 'REST_DAYS' and 'B2B' are assumed for a typical playoff game
    matchup_data = pd.DataFrame({
        'HOME': [1], # The current_home team is designated as home
        'REST_DAYS': [2], # Assuming 2 rest days for a playoff game
        'ROLLING_PTS': [rolling_pts_home],
        'ROLLING_WIN_PCT': [rolling_win_pct_home],
        'B2B': [0] # Assuming no back-to-back for a playoff game
    })

    # Ensure the columns match the features used for training
    matchup_data = matchup_data[features]

    # Predict the probability of the home team winning
    prob_home_wins = model.predict_proba(matchup_data)[:, 1][0]

    return prob_home_wins


# ------------------------------------------------
# FUNCTION TO SIMULATE A BEST-OF-7 SERIES
# ------------------------------------------------

def simulate_series(
    home_team,
    away_team
):

    home_wins = 0
    away_wins = 0

    game_number = 1

    print("\n======================================")
    print(f"{home_team} vs {away_team}")
    print("BEST-OF-7 SERIES")
    print("======================================\n")

    # 2-2-1-1-1 NBA playoff home format
    # 1 = home_team home court
    # 0 = away_team home court

    home_schedule = [1,1,0,0,1,0,1]

    for home_game in home_schedule:

        # Series already over
        if home_wins == 4 or away_wins == 4:
            break

        # Determine actual home/away
        if home_game == 1:

            current_home = home_team
            current_away = away_team

        else:

            current_home = away_team
            current_away = home_team

        # Predict home team win probability
        prob = predict_matchup(
            current_home,
            current_away
        )

        # Randomly simulate outcome
        winner = np.random.choice(
            [current_home,current_away],
            p=[prob,1-prob]
        )

        # Update series score
        if winner == home_team:
            home_wins += 1
        else:
            away_wins += 1

        # Display game result
        print(f"Game {game_number}: {winner} wins")

        print(
            f"Series Score: "
            f"{home_team} {home_wins} - "
            f"{away_team} {away_wins}"
        )

        print("----------------------------------")

        game_number += 1

    # Final winner
    if home_wins > away_wins:
        champion = home_team
    else:
        champion = away_team

    print("\n======================================")
    print(f"SERIES WINNER: {champion}")
    print("======================================\n")

    return champion


# ============================================================
# ROUND 2 MATCHUPS
# ============================================================

round2 = [

    ('CLE','DET'),
    ('OKC','LAL'),
    ('MIN','SAS'),
    ('NYK','PHI')

]


# ============================================================
# SIMULATE ROUND 2
# ============================================================

round2_winners = []

print("\n######################################")
print("ROUND 2 PLAYOFF SIMULATIONS")
print("######################################")

for matchup in round2:

    winner = simulate_series(
        matchup[0],
        matchup[1]
    )

    round2_winners.append(winner)


# ============================================================
# CONFERENCE FINALS
# ============================================================

east_final = (
    round2_winners[0],
    round2_winners[3]
)

west_final = (
    round2_winners[1],
    round2_winners[2]
)

print("\n######################################")
print("CONFERENCE FINALS")
print("######################################")

east_winner = simulate_series(
    east_final[0],
    east_final[1]
)

west_winner = simulate_series(
    west_final[0],
    west_final[1]
)


# ============================================================
# NBA FINALS
# ============================================================

print("\n######################################")
print("NBA FINALS")
print("######################################")

nba_champion = simulate_series(
    east_winner,
    west_winner
)

print("\n🏆 NBA CHAMPION:", nba_champion)


######################################
ROUND 2 PLAYOFF SIMULATIONS
######################################

CLE vs DET
BEST-OF-7 SERIES

Game 1: DET wins
Series Score: CLE 0 - DET 1
----------------------------------
Game 2: CLE wins
Series Score: CLE 1 - DET 1
----------------------------------
Game 3: DET wins
Series Score: CLE 1 - DET 2
----------------------------------
Game 4: CLE wins
Series Score: CLE 2 - DET 2
----------------------------------
Game 5: DET wins
Series Score: CLE 2 - DET 3
----------------------------------
Game 6: CLE wins
Series Score: CLE 3 - DET 3
----------------------------------
Game 7: CLE wins
Series Score: CLE 4 - DET 3
----------------------------------

SERIES WINNER: CLE


OKC vs LAL
BEST-OF-7 SERIES

Game 1: OKC wins
Series Score: OKC 1 - LAL 0
----------------------------------
Game 2: OKC wins
Series Score: OKC 2 - LAL 0
----------------------------------
Game 3: OKC wins
Series Score: OKC 3 - LAL 0
----------------------------------
Game 4: LAL 